In [0]:
%load_ext autoreload
%autoreload 2

# Check SNAG preprocessed features shape and annotation

In [0]:
import numpy as np
import os
import glob

def inspect_npy_shape(feature_dirs, num_samples=3):
    """
    Checks the shape of .npy files in the given directories.
    Args:
        feature_dirs (list): List of paths to feature folders (e.g. [rgb_dir, flow_dir])
    """
    for f_dir in feature_dirs:
        if not f_dir or not os.path.exists(f_dir):
            print(f"Skipping invalid path: {f_dir}")
            continue

        print(f"\n--- Inspecting Directory: {os.path.basename(f_dir)} ---")
        # Find all .npy files
        files = glob.glob(os.path.join(f_dir, "*.npy"))
        if not files:
            print("No .npy files found.")
            continue
            
        print(f"Found {len(files)} files. Sampling first {num_samples}:")
        
        for i, f_path in enumerate(files[:num_samples]):
            try:
                # Load the file
                feat = np.load(f_path)
                fname = os.path.basename(f_path)
                
                # Check properties
                shape = feat.shape
                dtype = feat.dtype
                
                print(f"\n  File: {fname}")
                print(f"    Shape: {shape} (Time Steps, Channels)")
                print(f"    Dtype: {dtype}")
                
                # Heuristic: Check against common config values
                # If shape is (T, 1024), it's likely just RGB or just Flow.
                # If shape is (T, 2048), it's likely pre-concatenated I3D.
                if shape[1] == 1024:
                    print("    > Likely I3D-RGB or I3D-Flow (1024 dims)")
                elif shape[1] == 2048:
                    print("    > Likely Concatenated I3D (2048 dims)")
                
                # Estimate Duration (Assuming standard stride 4 and 24fps)
                # T * 4 / 24 = Seconds
                est_sec = (shape[0] * 4) / 24.0
                print(f"    > Est. Duration (if stride=4, fps=24): {est_sec:.2f} seconds")
                
            except Exception as e:
                print(f"    Error loading {fname}: {e}")

# --- RUN THIS ---
rgb_path = "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/kinetics/rgb"
flow_path = "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/kinetics/flow"

inspect_npy_shape([rgb_path, flow_path])

In [0]:
import json

with open("/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta.json", "r") as f:
    json_data = json.load(f)

train_data = json_data["train"]
test_data = json_data["test"]

In [0]:
train_data["004QE"]

In [0]:
train_data["00HFP"]

In [0]:
test_data["00607"]

# Check Custom Dataset Loader for SNAG features

In [0]:
!pip install torch

In [0]:
from build_charades_sta_snag import *

In [0]:
import torch
from torch.utils.data import DataLoader
from functools import partial

import os

# --- Imports from your project ---
# Make sure these point to where you saved the classes!
# from your_script import CharadesSnagAdapter, snag_fixed_collate 
# (Assuming the adapter and collate are in the same file or imported)
glove_tokenizer = GloVeTokenizer()

In [0]:

# ==========================================
# 1. CONFIGURATION
# ==========================================
paths = {
    # CHANGE THESE to your actual paths
    "anno_file": "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta.json",
    "vid_feat_dir" : ["/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/rgb", "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/flow"]
    # "feat_rgb": "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/rgb",
    # "feat_flow": "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/flow",
}

# Standard SnAG / 2D-TAN Configuration for Fair Comparison
dataset_cfg = {
    "split": "train",
    "max_vid_len": 256,      # Fixed length
    "to_fixed_len": True,    # Force resizing
    "clip_size": 16,         # These values depend on your feature extraction
    "clip_stride": 4,
    "max_text_len": 16,
    
}



# ==========================================
# 3. VERIFICATION LOOP
# ==========================================
def test_one_batch():
    print(f">>> Initializing Adapter Dataset...")
    
    # Instantiate your Adapter
    dataset = CharadesSnagAdapter(
        is_training=True,
        tokenizer=glove_tokenizer,
        anno_file=paths["anno_file"],
        # feat_rgb=paths["feat_rgb"], # Adapters might need specific kwarg names depending on base class
        # feat_flow=paths["feat_flow"],
        vid_feat_dir=paths["vid_feat_dir"], 
        **dataset_cfg
        
    )

    # Create Loader
    loader = DataLoader(
        dataset, 
        batch_size=8, 
        collate_fn=snag_custom_collate, 
        shuffle=True
    )
    
    print(f">>> Fetching one batch...")
    number= 0
    for idx, batch in enumerate(loader):
        if batch is None: continue
        if number == 100:
            break
        # --- A. PRINT SHAPES ---
        vid_shape = batch['video_emb'].shape
        text_shape = batch['query_tokens'].shape
        #target_shape = batch['targets'].shape
        
        print("\n" + "="*40)
        print("          DATASET INSPECTION")
        print("="*40)
        print(f"1. Video Shape:  {vid_shape}")
        print(f"2. Text Shape:   {text_shape}")
        print(batch['query_tokens'][0])
        print(batch["query"])
        print(batch["query_mask"])
        #print(f"3. Target Shape: {target_shape}")
        print(f"4. Video IDs:    {batch['video_id']}")
        
        # --- B. AUTOMATED CORRECTNESS CHECKS ---
        print("\n" + "-"*40)
        print("          CORRECTNESS CHECKS")
        print("-"*40)
        
        # Check 1: Input Dimensions (B, 256, 2048)
        assert vid_shape[1] == 256, \
            f"❌ FAIL: Video length is {vid_shape[1]}, expected 256 (Fixed Length)."
        print("✅ Video Length is 256.")

        assert vid_shape[2] == 2048, \
            f"❌ FAIL: Feature dim is {vid_shape[2]}, expected 2048 (I3D RGB+Flow)."
        print("✅ Feature Dimensions are 2048.")

        # Check 2: Targets are within grid
        # Targets should be indices between 0.0 and 256.0
        # t_min, t_max = batch['targets'].min().item(), batch['targets'].max().item()
        # print(f"ℹ️  Targets Range: [{t_min:.2f}, {t_max:.2f}]")
        
        # assert t_max <= 256.0, \
        #     f"❌ FAIL: Targets exceed max length! Found {t_max}."
        # print("✅ Targets are within grid bounds (<= 256).")
        
        # Check 3: Check Mask Logic
        # For fixed length, mask should be all True
        mask_integrity = batch['video_mask'].all().item()
        if mask_integrity:
            print("✅ Video Mask is all True (Correct for Fixed Length).")
        else:
            print("⚠️ WARNING: Video Mask has False values (Unexpected for to_fixed_len=True).")

        print("\n>>> TEST PASSED SUCCESSFULLY.")
        

def verify_timestamps():
    print(f">>> Initializing...")
    dataset = CharadesSnagAdapter(
        tokenizer=glove_tokenizer,
        anno_file=paths["anno_file"],
        vid_feat_dir=paths["vid_feat_dir"],
        **dataset_cfg
    )
    loader = DataLoader(dataset, batch_size=4, collate_fn=snag_custom_collate, shuffle=True)
    
    print(f">>> Comparing Calculated vs. Ground Truth...\n")
    
    for batch in loader:
        #targets = batch['targets']     # The grid indices (e.g. 45.5)
        durations = batch['duration'] # The video duration (e.g. 30.0)
        #text_ids = batch['text_ids']   # Keys to look up GT
        print(batch.keys())
        for i in range(len(batch["video_id"])):
            # 1. Get Values
            #idx_start, idx_end = targets[i][0].item(), targets[i][1].item()
            idx_start, idx_end = batch['i0'][i].item(), batch['i1'][i].item()
            duration = durations[i]
            
            # 2. METHOD 1 CALCULATION (Index -> Seconds)
            # Formula: (Index / 256) * Duration
            calc_start = (idx_start / 256.0) * duration
            calc_end = (idx_end / 256.0) * duration
            
            # 3. Get Ground Truth from Dataset
            # gt_segment = dataset.text_dict[text_ids[i]]['segment']
            # # --- FIX: Ensure it is a flat list/array ---
            # # If it comes out as [[s, e]], flatten it to [s, e]
            # gt_segment = np.array(gt_segment).reshape(-1)
            gt_start, gt_end = batch["start_sec"][i], batch["end_sec"][i]
            
            # 4. Print Comparison
            print(f"Sample {i}: {batch["video_id"][i]}")
            print(f"  Duration:      {duration:.2f}s")
            print(f"  Target Indices: [{idx_start:.2f}, {idx_end:.2f}] (Grid 0-256)")
            print(f"  Calculated Time: {calc_start:.2f}s - {calc_end:.2f}s")
            print(f"  GT Timestamp:    {gt_start:.2f}s - {gt_end:.2f}s")
            
            # 5. Check Error
            diff = abs(calc_start - gt_start)
            if diff < 0.5:
                print("  ✅ MATCH (Difference < 0.5s)")
            else:
                print(f"  ⚠️ MISMATCH (Diff: {diff:.2f}s) - Likely due to rounding/windowing")
            print("-" * 30)
            
        break
if __name__ == "__main__":
    test_one_batch()
    verify_timestamps()

# Verify converting fixed length video index to actual timestamp

In [0]:
def indices_to_timestamps_fixed(pred_idx, duration, max_len=256):
    """
    Converts fixed-length grid indices to seconds.
    Args:
        pred_idx (float): The index predicted by the model (e.g., 45.5)
        duration (float): The original video duration in seconds.
        max_len (int): The fixed length used during training (e.g., 256).
    """
    # Clamp index to [0, max_len] to avoid predicting negative time or > duration
    pred_idx = min(max(0, pred_idx), max_len)
    
    time_sec = (pred_idx / max_len) * duration
    return time_sec

In [0]:
indices_to_timestamps_fixed(158, 19.08)

In [0]:
from time import sleep
sleep(600)

In [0]:
glove_tokenizer = GloVeTokenizer()

In [0]:
from build_charades_sta_snag import *
paths = {
"anno_root": "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations",
"vid_feat_dir" : ["/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/rgb", "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/i3d_features/charades/flow"]
}

# Standard SnAG Configuration for Fair Comparison
train_dataset_cfg = {
    "split": "train",
    "max_vid_len": 256,      # Fixed length
    "to_fixed_len": True,    # Force resizing
    "clip_size": 16,         # These values depend on your feature extraction
    "clip_stride": 4,
    "max_text_len": 16,
    
}

# Standard SnAG Configuration for Fair Comparison
val_dataset_cfg = {
    "split": "val",
    "max_vid_len": 256,      # Fixed length
    "to_fixed_len": True,    # Force resizing
    "clip_size": 16,         # These values depend on your feature extraction
    "clip_stride": 4,
    "max_text_len": 16,
    
}

# Standard SnAG Configuration for Fair Comparison
test_dataset_cfg = {
    "split": "test",
    "max_vid_len": 256,      # Fixed length
    "to_fixed_len": True,    # Force resizing
    "clip_size": 16,         # These values depend on your feature extraction
    "clip_stride": 4,
    "max_text_len": 16,
    
}

# Instantiate your Adapter
train_ds = CharadesSnagAdapter(
    is_training=True,
    tokenizer=glove_tokenizer,
    anno_file=f"{paths["anno_root"]}/charades_sta_train_split.json",
    vid_feat_dir=paths["vid_feat_dir"], 
    **train_dataset_cfg
    
)
val_ds = CharadesSnagAdapter(
    is_training=False,
    tokenizer=glove_tokenizer,
    anno_file=f"{paths["anno_root"]}/charades_sta_val_split.json",
    vid_feat_dir=paths["vid_feat_dir"], 
    **val_dataset_cfg
    
)
test_ds = CharadesSnagAdapter(
    is_training=False,
    tokenizer=glove_tokenizer,
    anno_file=f"{paths["anno_root"]}/charades_sta_test_split.json",
    vid_feat_dir=paths["vid_feat_dir"], 
    **test_dataset_cfg
    
)

# Create Loader
tr_dataloader = DataLoader(
    train_ds, 
    batch_size=16, 
    collate_fn=snag_custom_collate, 
    shuffle=True,
    drop_last=True
)
val_dataloader = DataLoader(
    val_ds, 
    batch_size=16, 
    collate_fn=snag_custom_collate, 
    shuffle=False
)
test_dataloader = DataLoader(
    test_ds, 
    batch_size=16, 
    collate_fn=snag_custom_collate, 
    shuffle=False
)

In [0]:
print(len(train_ds),
len(val_ds),
len(test_ds))

# Split annoation file into train, val, test annotation files

In [0]:
import json
import random
import os

def split_charadessta_train_val(
    train_json_path: str,
    out_train_path: str,
    out_val_path: str,
    out_test_path: str,
    val_ratio: float = 0.1,
    seed: int = 123,
):
    """
    Splits a Charades-STA formatted JSON (nested dict) into Train and Val sets.
    Input Format: {"train": {"VID1": {...}, ...}, "test": {...}}
    Output Format: {"VID1": {...}, "VID2": {...}} (Direct dictionary for Loader)
    """
    random.seed(seed)
    
    print(f"Loading data from {train_json_path}...")
    with open(train_json_path, "r") as f:
        full_data = json.load(f)

    # 1. Access the 'train' split specifically
    if "train" not in full_data:
        raise ValueError(f"Input file must contain a 'train' key. Found: {full_data.keys()}")
    
    # train_data is: {"VID": {"duration": X, "annotations": [...]}, ...}
    train_data = full_data["train"]
    test_data = full_data["test"]
    
    # 2. Get Video IDs and Shuffle
    vids = list(train_data.keys())
    random.shuffle(vids)

    # 3. Calculate Split Index
    n_val = max(1, int(len(vids) * val_ratio))
    val_vids = set(vids[:n_val])
    train_vids = set(vids[n_val:])

    # 4. Create New Dictionaries
    # We preserve the exact structure of the value (duration, annotations, etc.)
    new_train_split = {"train" : {vid: train_data[vid] for vid in train_vids}}
    new_val_split = {"val" : {vid: train_data[vid] for vid in val_vids}}
    test_split = {"test" : test_data}

    # 5. Save outputs
    # We save them as flat dictionaries of videos so the Dataloader can iterate .items()
    os.makedirs(os.path.dirname(out_train_path), exist_ok=True)
    os.makedirs(os.path.dirname(out_val_path), exist_ok=True)

    with open(out_train_path, "w") as f:
        json.dump(new_train_split, f, indent=2)
        
    with open(out_val_path, "w") as f:
        json.dump(new_val_split, f, indent=2)

    with open(out_test_path, "w") as f:
        json.dump(test_split, f, indent=2)

    # 6. Statistics
    # Calculate sample counts by summing annotation lists
    n_train_samples = sum(len(v["annotations"]) for v in new_train_split["train"].values())
    n_val_samples = sum(len(v["annotations"]) for v in new_val_split["val"].values())
    n_test_samples = sum(len(v["annotations"]) for v in test_split["test"].values())

    print(f"--- Split Complete ---")
    print(f"Original Train Videos: {len(vids)}")
    print(f"New Train: {len(new_train_split["train"])} videos ({n_train_samples} queries) -> Saved to {out_train_path}")
    print(f"New Val:   {len(new_val_split["val"])} videos ({n_val_samples} queries) -> Saved to {out_val_path}")
    print(f"Test:   {len(test_split["test"])} videos ({n_test_samples} queries) -> Saved to {out_test_path}")

# Example Usage:
split_charadessta_train_val(
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta.json", 
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta_train_split.json", 
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta_val_split.json",
    "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta_test_split.json"
)

Sanity Check, comparing CLIP splits

In [0]:
json_list = "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/charades_sta_val_split.json"

with open(json_list, "r") as f:
    data = json.load(f)
vid_list = []
for item in data:
    if item["video_id"] not in vid_list:
        vid_list.append(item["video_id"])

In [0]:
json_list = "/Volumes/biomedicalinformatics_analytics/dev_lab_johnson/open_source_video_datasets/Charades-STA/snag/charades_sta/annotations/charades_sta_val_split.json"
with open(json_list, "r") as f:
    data = json.load(f)
new_vid_list = []
for k,v in data["val"].items():
    if k not in vid_list:
        print("does not exist")
    else:
        new_vid_list.append(k)

In [0]:
len(new_vid_list)

In [0]:
len(vid_list)

In [0]:
sorted(new_vid_list)

In [0]:
sorted(vid_list)